# 05 XGBoost E + Logistic Regression E 50:50 앙상블

팀장 요청: 저장된 OOF/Test 확률을 0.5씩 평균, 팀 공통 Fold로 평가, `save_predictions`로 저장. **재학습 없음.**

주의: 같은 학습/테스트 행 순서, 같은 클래스 열 순서, 같은 검증 Fold로 생성된 예측이어야 합니다. 저장 메타에 이 정보가 모두 기록되지는 않으므로 최종 확인이 필요합니다.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

start = Path.cwd().resolve()
ROOT = next((p for p in (start, *start.parents) if (p / "src" / "config.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError(f"팀 저장소의 src/config.py를 찾지 못했습니다. 현재 경로: {start}")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import DATA_DIR, EXPERIMENT_DIR
from src.data_loader import load_data, split_xy, encode_target
from src.validation import get_folds
from src.evaluation import save_predictions, log_result

PRED_DIR = EXPERIMENT_DIR / "preds"
print("저장소:", ROOT)
print("예측 파일 폴더:", PRED_DIR)
assert PRED_DIR.is_dir(), "experiments/preds 폴더가 없습니다. 최신 dev를 확인하세요."


저장소: C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype
예측 파일 폴더: C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\experiments\preds


In [2]:
# 메타 이름으로 파일을 정확히 찾습니다. Discord에서 밑줄이 사라져도 안전합니다.
XGB_NAME = "A_model__xgb_E_group_1109_depth3"
LR_NAME = "H_data__E_aa_pairs_truncation_group_lr"

def load_pair(name):
    meta_path = PRED_DIR / f"{name}__meta.json"
    oof_path = PRED_DIR / f"{name}__oof.npy"
    test_path = PRED_DIR / f"{name}__test.npy"
    missing = [str(p.name) for p in (meta_path, oof_path, test_path) if not p.is_file()]
    if missing:
        print("사용 가능한 예측 메타 파일:")
        for p in sorted(PRED_DIR.glob("*__meta.json")):
            print(" -", p.name)
        raise FileNotFoundError(f"{name}: 누락된 파일 {missing}. 위 목록을 보고 정확한 이름을 설정하세요.")
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    return np.load(oof_path), np.load(test_path), meta

xgb_oof, xgb_test, xgb_meta = load_pair(XGB_NAME)
lr_oof, lr_test, lr_meta = load_pair(LR_NAME)
for name, a, b, m in [("XGB",xgb_oof,xgb_test,xgb_meta),("LR",lr_oof,lr_test,lr_meta)]:
    print(name, "OOF:", a.shape, "Test:", b.shape, "meta classes:", m.get("classes",[])[:4], "...")


XGB OOF: (6201, 26) Test: (2546, 26) meta classes: ['ACC', 'BLCA', 'BRCA', 'CESC'] ...
LR OOF: (6201, 26) Test: (2546, 26) meta classes: ['0', '1', '2', '3'] ...


In [3]:
# 정답과 Fold는 train 데이터로만 생성합니다.
train, test = load_data(DATA_DIR)
X_raw, y_raw = split_xy(train)
y, le = encode_target(y_raw)
folds = get_folds(y, X=X_raw, classes=le.classes_)
K = len(le.classes_)
expected_oof = (len(y), K)
expected_test = (len(test), K)

for name, oof, pred, meta in [("XGB", xgb_oof, xgb_test, xgb_meta), ("LR", lr_oof, lr_test, lr_meta)]:
    assert oof.shape == expected_oof, f"{name}: OOF 크기 불일치 {oof.shape} != {expected_oof}"
    assert pred.shape == expected_test, f"{name}: Test 크기 불일치 {pred.shape} != {expected_test}"
    for a, label in ((oof,"OOF"),(pred,"Test")):
        assert np.isfinite(a).all(), f"{name}: {label} NaN/Inf 발견"
        assert np.all(a >= -1e-6) and np.all(a <= 1+1e-6), f"{name}: {label} 확률 범위 오류"
        assert np.allclose(a.sum(axis=1),1.0,atol=1e-3), f"{name}: {label} 확률 합이 1이 아님"
    assert meta.get("n_train") == len(y) and meta.get("n_test") == len(test), f"{name}: 메타 행 수 불일치"

# XGB 메타는 원래 클래스 이름. LR 메타는 팀장 확인대로 정수 인코딩 0..K-1.
expected_names = [str(c) for c in le.classes_]
expected_numbers = [str(i) for i in range(K)]
assert xgb_meta.get("classes") == expected_names, "XGB 메타 클래스 순서가 le.classes_와 다릅니다"
assert lr_meta.get("classes") in (expected_names, expected_numbers), "LR 메타 클래스 정의가 예상과 다릅니다"
if lr_meta["classes"] == expected_numbers:
    print("주의: LR 메타는 숫자 클래스입니다. 팀장님이 열 순서가 le.classes_와 같다고 확인한 전제에 따라 사용합니다.")
print("확률 형태, 값, 클래스 메타 검사 완료")
print("중요: 메타만으로 train/test 행 순서와 LR Fold 동일성을 증명할 수는 없습니다. 원본 노트북/팀장 확인 필요.")


주의: LR 메타는 숫자 클래스입니다. 팀장님이 열 순서가 le.classes_와 같다고 확인한 전제에 따라 사용합니다.
확률 형태, 값, 클래스 메타 검사 완료
중요: 메타만으로 train/test 행 순서와 LR Fold 동일성을 증명할 수는 없습니다. 원본 노트북/팀장 확인 필요.


In [4]:
# 팀 공통 Fold별 점수 계산: 이미 만들어진 OOF로 평가하며 재학습하지 않습니다.
ens_oof = (xgb_oof.astype(np.float64) + lr_oof.astype(np.float64)) / 2
ens_test = (xgb_test.astype(np.float64) + lr_test.astype(np.float64)) / 2
ens_label = ens_oof.argmax(axis=1)

scores = []
seen = np.zeros(len(y), dtype=np.int8)
for i, (tr_idx, va_idx) in enumerate(folds, 1):
    assert len(np.intersect1d(tr_idx,va_idx)) == 0
    assert not seen[va_idx].any()
    seen[va_idx] += 1
    score = f1_score(y[va_idx], ens_label[va_idx], average="macro")
    scores.append(float(score))
    print(f"Fold {i}: Macro F1 = {score:.4f}")
assert np.all(seen == 1)
cv = {"scores":scores,"mean":float(np.mean(scores)),"std":float(np.std(scores)),
      "oof":ens_label,"oof_proba":ens_oof}
overall = f1_score(y, ens_label, average="macro")
print(f"CV Macro F1: {cv['mean']:.4f} ± {cv['std']:.4f}")
print(f"전체 OOF Macro F1: {overall:.4f} (Fold 평균과 다른 지표)")
print("팀장 기대 CV 약 0.510; 차이가 크면 저장하지 말고 Fold/클래스/행 순서를 확인하세요.")


Fold 1: Macro F1 = 0.4916
Fold 2: Macro F1 = 0.5224
Fold 3: Macro F1 = 0.4922
Fold 4: Macro F1 = 0.5258
Fold 5: Macro F1 = 0.5163
CV Macro F1: 0.5096 ± 0.0148
전체 OOF Macro F1: 0.5113 (Fold 평균과 다른 지표)
팀장 기대 CV 약 0.510; 차이가 크면 저장하지 말고 Fold/클래스/행 순서를 확인하세요.


In [5]:
# 저장 전 확인: 팀장 기대값과 크게 다르면 실행을 중단합니다.
# 0.510은 참고값이며 실제 점수는 출력된 Fold 결과를 우선합니다.
assert abs(cv["mean"] - 0.510) < 0.02, (
    f"기대 CV 0.510과 차이가 큽니다: {cv['mean']:.4f}. 저장 전 Fold/행/클래스 순서를 검증하세요."
)
NAME = "ensemble_E_xgb_lr_5050_group_1109"
assert not (PRED_DIR / f"A_model__{NAME}__meta.json").exists(), "같은 앙상블 이름의 저장 파일이 이미 있습니다. 덮어쓰지 않습니다."

# 로그는 담당자 파일에만 추가합니다.
log_result(owner="A_model", notebook="05_ensemble_E_xgb_lr.ipynb",
           experiment=NAME, features="E_9174", model="ProbabilityAverage_0.5_0.5",
           params={"XGB":XGB_NAME,"LR":LR_NAME,"weights":[0.5,0.5]}, cv=cv,
           notes="OOF/Test arithmetic mean; no retraining; LR numeric meta class order confirmed by leader")
save_predictions(owner="A_model", name=NAME,
                 oof_proba=ens_oof, test_proba=ens_test,
                 classes=le.classes_, cv=cv,
                 notes="XGB+LR E 50:50; identical row/class order assumed from source runs; no retraining")
print("앙상블 저장 완료. PR에는 이 노트북, A_model 로그, 생성된 3개 예측 파일만 올리세요.")


기록: log_A_model.csv ← ensemble_E_xgb_lr_5050_group_1109 (CV 0.5096)
예측 저장: A_model__ensemble_E_xgb_lr_5050_group_1109 (CV 0.5096) → experiments/preds/
앙상블 저장 완료. PR에는 이 노트북, A_model 로그, 생성된 3개 예측 파일만 올리세요.
